In [6]:
import os, time, requests, pandas as pd
from dataclasses import dataclass, asdict
from urllib.parse import urljoin

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# -------------------- site config --------------------
SITE_ROOT = "https://expinterweb.mites.gob.es"
BASE_URL = f"{SITE_ROOT}/regcon/pub/consultaPublicaEstatal"

def make_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--window-size=1920,1080")
    
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=opts)
    
    driver.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
        "source": "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
    })
    return driver

# -------------------- small utils --------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def append_rows(out_dir, rows):
    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    df = pd.DataFrame([asdict(r) if hasattr(r, "__dict__") else r for r in rows])
    mode, header = ("a", False) if os.path.exists(csv_path) else ("w", True)
    df.to_csv(csv_path, mode=mode, header=header, index=False, encoding="utf-8")

# -------------------- parsing --------------------
@dataclass
class AgreementSpain:
    codigo: str | None
    denominacion: str | None
    tipo_tramite: str | None
    autoridad_laboral: str | None
    vigencia_desde: str | None
    vigencia_hasta: str | None
    detalle_url: str | None
    pdf_url: str | None
    pdf_saved_as: str | None

def download_pdf(url, out_dir, agreement_id):
    if not url: return None
    ensure_dir(out_dir)
    name = f"{agreement_id}.pdf" 
    path = os.path.join(out_dir, name)
    if os.path.exists(path): return path 
        
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        r = requests.get(url, headers=headers, timeout=60)
        r.raise_for_status()
        if "application/pdf" in r.headers.get("Content-Type", "") or r.content.startswith(b"%PDF"):
            with open(path, "wb") as f:
                f.write(r.content)
            return path
        return None
    except Exception as e:
        print(f"Failed to download PDF {url}: {e}")
        return None

# -------------------- main scraper --------------------
def scrape_spain(out_dir="spain_data", target_count=10, pause=2.0):
    ensure_dir(out_dir)
    pdf_dir = os.path.join(out_dir, "pdfs")
    ensure_dir(pdf_dir)

    d = make_driver(headless=False) 
    collected_agreements = []
    
    date_ranges = [("01/01/2023", "31/01/2023")]

    try:
        for date_from, date_to in date_ranges:
            print(f"\n--- Loading {BASE_URL} ---")
            d.get(BASE_URL)
            time.sleep(3) 
            
            print(f"Entering date filter: {date_from} to {date_to}...")
            
            # 1. Type into 'Desde' box using exact ID
            box_desde = WebDriverWait(d, 10).until(
                EC.element_to_be_clickable((By.ID, "fhIncripcionPublicacionDesde"))
            )
            box_desde.clear()
            box_desde.send_keys(date_from)

            # 2. Type into 'Hasta' box using exact ID
            box_hasta = d.find_element(By.ID, "fhIncripcionPublicacionHasta")
            box_hasta.clear()
            box_hasta.send_keys(date_to)

            # 3. Click the Search button using exact ID
            print("Clicking search button...")
            search_btn = d.find_element(By.ID, "_buscar")
            d.execute_script("arguments[0].click();", search_btn)

            # 4. Wait for table
            print("Waiting for the results table to load...")
            try:
                WebDriverWait(d, 15).until(EC.presence_of_element_located((By.XPATH, "//table//tr/td")))
                print("Table loaded successfully!")
            except:
                print("Timeout waiting for table. Site might be slow or no results found.")
                continue 
            
            # 5. Parse rows
            rows = d.find_elements(By.XPATH, "//table//tr[td]")
            print(f"Found {len(rows)} agreements.")
            
            for row in rows[:target_count]:
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) < 7: continue
                
                codigo = cols[0].text.strip()
                detalle_link = None
                try:
                    action_a = cols[-1].find_element(By.TAG_NAME, "a")
                    detalle_link = action_a.get_attribute("href")
                except: pass

                ag = AgreementSpain(
                    codigo=codigo, denominacion=cols[1].text.strip(),
                    tipo_tramite=cols[2].text.strip(), autoridad_laboral=cols[3].text.strip(),
                    vigencia_desde=cols[4].text.strip(), vigencia_hasta=cols[5].text.strip(),
                    detalle_url=detalle_link, pdf_url=None, pdf_saved_as=None
                )
                collected_agreements.append(ag)

            # 6. Fetch PDFs
            for ag in collected_agreements:
                # MOVE THIS LINE HERE: Save the metadata FIRST, no matter what!
                append_rows(out_dir, [ag])
                
                if not ag.detalle_url: 
                    print(f"No detail link found for {ag.codigo}, skipping PDF download.")
                    continue
                    
                print(f"Fetching PDF details for agreement {ag.codigo}...")
                d.get(ag.detalle_url)
                time.sleep(pause) 
                
                try:
                    pdf_a = d.find_element(By.XPATH, "//a[contains(@href, '.pdf') or contains(translate(text(), 'PDF', 'pdf'), 'pdf')]")
                    ag.pdf_url = pdf_a.get_attribute("href")
                    if ag.pdf_url:
                        ag.pdf_saved_as = download_pdf(ag.pdf_url, pdf_dir, ag.codigo)
                except:
                    print(f"Could not find PDF link for {ag.codigo}")


            if len(collected_agreements) >= target_count:
                break

        print(f"\nSuccessfully processed {len(collected_agreements)} agreements for prototype.")
        
    finally:
        d.quit()

    csv_path = os.path.join(out_dir, "spain_agreements.csv")
    try:
        return pd.read_csv(csv_path)
    except Exception:
        return pd.DataFrame()

# Run the prototype:
df = scrape_spain(target_count=10)
print(df.head())


--- Loading https://expinterweb.mites.gob.es/regcon/pub/consultaPublicaEstatal ---
Entering date filter: 01/01/2023 to 31/01/2023...
Clicking search button...
Waiting for the results table to load...
Table loaded successfully!
Found 15 agreements.
No detail link found for 11100131012015, skipping PDF download.
No detail link found for 71103602012024, skipping PDF download.
No detail link found for 36001155011981, skipping PDF download.
No detail link found for , skipping PDF download.
No detail link found for , skipping PDF download.
No detail link found for 82000882012006, skipping PDF download.
No detail link found for , skipping PDF download.
No detail link found for 90121882112023, skipping PDF download.
No detail link found for 90121893112023, skipping PDF download.
No detail link found for 90121872112023, skipping PDF download.

Successfully processed 10 agreements for prototype.
         codigo                                       denominacion  \
0  1.110013e+13   SERVICIO DE 